# EXP_v5: D-optimality + MAP (2D 2PL)

`EXP_v5/README.md` の実験条件に従い、2次元2PL項目バンクでD-optimalityによるCATを実行します。能力推定には全相関条件で共通の事前分布 $N(\mathbf{0}, \mathbf{I}_2)$ に基づくMAP推定を使用し、項目選択の情報行列にも事前情報 $\mathbf{I}_2$ を含めます。

既定では bank 1・$\rho=0.0$・5,000人・40問を評価します。条件は `Config` で変更できます。結果は受検者・ステップ単位の `records` と、ステップ単位の `summary` の両方を `EXP_v5/results/` に保存します。

In [1]:
# -*- coding: utf-8 -*-
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit


def find_project_root() -> Path:
    candidates: list[Path] = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "EXP_v5" / "README.md").is_file():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "EXP_v5" / "README.md").is_file():
            return root

    raise FileNotFoundError(
        "Could not find the project root containing EXP_v5/README.md."
    )


ROOT = find_project_root()
EXP_DIR = ROOT / "EXP_v5"
RESULTS_DIR = EXP_DIR / "results"

print(f"Project root: {ROOT}")
print(f"Results dir : {RESULTS_DIR}")

Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/results


In [2]:
@dataclass(frozen=True)
class Config:
    test_length: int = 40

    # Bank / evaluation data
    bank_id: int = 1
    rho: float = 0.0
    n_items: int = 150
    testing_size: int = 0  # 0: use all theta values
    theta_csv: str = ""  # empty: use the rho/bank-specific EXP_v5 file

    # Reproducibility / numerical settings
    seed: int = 20260430
    map_tolerance: float = 1e-8
    map_max_iterations: int = 50
    output_suffix: str = "_python"


RHO_TAGS = {0.0: "rho00", 0.3: "rho03", 0.6: "rho06"}
PRIOR_PRECISION = np.eye(2, dtype=np.float64)

## 2次元2PL・MAP推定・D-optimality

反応確率は $P_j(\boldsymbol{\theta})=\operatorname{logistic}(\mathbf{a}_j^\top\boldsymbol{\theta}-b_j)$ です。MAP推定では、負の対数尤度に $\frac{1}{2}\boldsymbol{\theta}^\top\boldsymbol{\theta}$ を加えます。事前分布により目的関数は強凸となるため、初期段階でも有限かつ一意な推定値が得られます。

候補項目の比較には行列式補題を使います。現在の事後情報行列を $\mathbf{J}$、候補項目の重みを $w=P(1-P)$ とすると、$\det(\mathbf{J}+w\mathbf{a}\mathbf{a}^\top)=\det(\mathbf{J})(1+w\mathbf{a}^\top\mathbf{J}^{-1}\mathbf{a})$ です。同一受検者内では $\det(\mathbf{J})$ が候補間で共通なので、log-determinant増分を最大化して選択します。

In [3]:
def response_probability(item_paras: np.ndarray, theta: np.ndarray) -> np.ndarray:
    a = item_paras[..., :2]
    b = item_paras[..., 2]
    return expit(np.sum(a * theta, axis=-1) - b)


def posterior_information(
    item_bank: np.ndarray,
    theta_current: np.ndarray,
    item_ids: np.ndarray,
) -> np.ndarray:
    testing_size = len(theta_current)
    information = np.broadcast_to(
        PRIOR_PRECISION, (testing_size, 2, 2)
    ).copy()
    if item_ids.shape[0] == 0:
        return information

    administered = item_bank[item_ids.T]
    a = administered[..., :2]
    b = administered[..., 2]
    eta = np.einsum("nti,ni->nt", a, theta_current) - b
    p = expit(eta)
    weight = p * (1.0 - p)
    information += np.einsum("nt,nti,ntj->nij", weight, a, a)
    return information


def choose_d_optimal(
    item_bank: np.ndarray,
    theta_current: np.ndarray,
    item_ids: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Select each examinee's unadministered item by D-optimality."""
    information = posterior_information(item_bank, theta_current, item_ids)
    information_inv = np.linalg.inv(information)

    a = item_bank[:, :2]
    b = item_bank[:, 2]
    eta = theta_current @ a.T - b
    p = expit(eta)
    weight = p * (1.0 - p)
    quadratic = np.einsum("jd,nde,je->nj", a, information_inv, a)
    logdet_gain = np.log1p(weight * quadratic)

    if item_ids.shape[0] > 0:
        subject_indices = np.arange(len(theta_current))[:, None]
        logdet_gain[subject_indices, item_ids.T] = -np.inf

    selected = logdet_gain.argmax(axis=1).astype(np.int64)
    selected_gain = logdet_gain[np.arange(len(theta_current)), selected]
    return selected, selected_gain

In [4]:
def map_objective(
    theta: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    responses: np.ndarray,
) -> np.ndarray:
    eta = np.einsum("nti,ni->nt", a, theta) - b
    negative_log_likelihood = np.sum(
        np.logaddexp(0.0, eta) - responses * eta, axis=1
    )
    negative_log_prior = 0.5 * np.sum(theta**2, axis=1)
    return negative_log_likelihood + negative_log_prior


def map_gradient(
    theta: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    responses: np.ndarray,
) -> np.ndarray:
    eta = np.einsum("nti,ni->nt", a, theta) - b
    return theta + np.einsum("nt,nti->ni", expit(eta) - responses, a)


def estimate_theta_map(
    item_bank: np.ndarray,
    item_ids: np.ndarray,
    responses: np.ndarray,
    current_theta: np.ndarray,
    tolerance: float,
    max_iterations: int,
) -> tuple[np.ndarray, np.ndarray, int]:
    """Estimate all examinees by damped Newton MAP optimization."""
    administered = item_bank[item_ids.T]
    a = administered[..., :2]
    b = administered[..., 2]
    response_by_subject = responses.T.astype(np.float64, copy=False)
    theta = current_theta.copy()
    armijo = 1e-4
    max_line_search_iterations = 25

    iterations_used = 0
    for iteration in range(1, max_iterations + 1):
        iterations_used = iteration
        eta = np.einsum("nti,ni->nt", a, theta) - b
        p = expit(eta)
        residual = p - response_by_subject
        gradient = theta + np.einsum("nt,nti->ni", residual, a)
        weight = p * (1.0 - p)
        hessian = np.broadcast_to(
            PRIOR_PRECISION, (len(theta), 2, 2)
        ).copy()
        hessian += np.einsum("nt,nti,ntj->nij", weight, a, a)
        newton_direction = np.linalg.solve(
            hessian, gradient[..., np.newaxis]
        ).squeeze(axis=-1)

        if np.max(np.abs(gradient)) <= tolerance:
            break

        current_objective = map_objective(
            theta, a, b, response_by_subject
        )
        directional_derivative = np.sum(
            gradient * newton_direction, axis=1
        )
        step_size = np.ones(len(theta), dtype=np.float64)

        for _ in range(max_line_search_iterations):
            candidate = theta - step_size[:, None] * newton_direction
            candidate_objective = map_objective(
                candidate, a, b, response_by_subject
            )
            accepted = candidate_objective <= (
                current_objective
                - armijo * step_size * directional_derivative
            )
            if np.all(accepted):
                break
            step_size[~accepted] *= 0.5

        update = step_size[:, None] * newton_direction
        theta -= update
        if np.max(np.abs(update)) <= tolerance:
            break

    final_gradient = map_gradient(theta, a, b, response_by_subject)
    converged = np.max(np.abs(final_gradient), axis=1) <= max(
        100.0 * tolerance, 1e-6
    )
    return theta, converged, iterations_used

In [5]:
def safe_correlation(x: np.ndarray, y: np.ndarray) -> float:
    if np.std(x, ddof=1) == 0 or np.std(y, ddof=1) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def summarize_steps(
    theta_true: np.ndarray, theta_history: np.ndarray
) -> pd.DataFrame:
    rows: list[dict[str, float | int]] = []
    for step, theta_est in enumerate(theta_history, start=1):
        error = theta_est - theta_true
        rows.append(
            {
                "step": step,
                "Bias_theta1": float(np.mean(error[:, 0])),
                "RMSE_theta1": float(np.sqrt(np.mean(error[:, 0] ** 2))),
                "MAE_theta1": float(np.mean(np.abs(error[:, 0]))),
                "r_theta1": safe_correlation(
                    theta_true[:, 0], theta_est[:, 0]
                ),
                "Bias_theta2": float(np.mean(error[:, 1])),
                "RMSE_theta2": float(np.sqrt(np.mean(error[:, 1] ** 2))),
                "MAE_theta2": float(np.mean(np.abs(error[:, 1]))),
                "r_theta2": safe_correlation(
                    theta_true[:, 1], theta_est[:, 1]
                ),
                "RMSE_overall": float(np.sqrt(np.mean(error**2))),
            }
        )
    return pd.DataFrame(rows)


def run_d_optimality(
    cfg: Config, item_bank: np.ndarray, theta_true: np.ndarray
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(cfg.seed)
    testing_size = len(theta_true)

    theta_current = np.zeros((testing_size, 2), dtype=np.float64)
    item_ids = np.empty((0, testing_size), dtype=np.int64)
    responses = np.empty((0, testing_size), dtype=np.int64)
    theta_history = np.empty((0, testing_size, 2), dtype=np.float64)
    selection_gain_history = np.empty(
        (0, testing_size), dtype=np.float64
    )

    for step in range(cfg.test_length):
        selected, selected_gain = choose_d_optimal(
            item_bank, theta_current, item_ids
        )
        probability = response_probability(
            item_bank[selected], theta_true
        )
        step_responses = (rng.random(testing_size) <= probability).astype(
            np.int64
        )

        item_ids = np.concatenate((item_ids, selected[np.newaxis, :]))
        responses = np.concatenate(
            (responses, step_responses[np.newaxis, :])
        )
        selection_gain_history = np.concatenate(
            (selection_gain_history, selected_gain[np.newaxis, :])
        )
        theta_current, converged, iterations_used = estimate_theta_map(
            item_bank,
            item_ids,
            responses,
            theta_current,
            tolerance=cfg.map_tolerance,
            max_iterations=cfg.map_max_iterations,
        )
        if not np.all(converged):
            failed_count = int(np.count_nonzero(~converged))
            raise RuntimeError(
                f"MAP optimization did not converge for {failed_count} "
                f"examinees at step {step + 1}."
            )

        theta_history = np.concatenate(
            (theta_history, theta_current[np.newaxis, :, :])
        )
        error = theta_current - theta_true
        print(
            f"step {step + 1:2d}, "
            f"rmse_theta1 {np.sqrt(np.mean(error[:, 0] ** 2)):.3f}, "
            f"rmse_theta2 {np.sqrt(np.mean(error[:, 1] ** 2)):.3f}, "
            f"rmse_overall {np.sqrt(np.mean(error**2)):.3f}, "
            f"map_iterations {iterations_used}"
        )

    user_id_col = np.repeat(np.arange(1, testing_size + 1), cfg.test_length)
    step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size)
    error_history = theta_history - theta_true[np.newaxis, :, :]
    records = pd.DataFrame(
        {
            "userID": user_id_col,
            "step": step_col,
            "itemID": (item_ids + 1).T.reshape(-1),
            "resp": responses.T.reshape(-1),
            "theta1_true": np.repeat(theta_true[:, 0], cfg.test_length),
            "theta2_true": np.repeat(theta_true[:, 1], cfg.test_length),
            "theta1_est": theta_history[:, :, 0].T.reshape(-1),
            "theta2_est": theta_history[:, :, 1].T.reshape(-1),
            "bias_theta1": error_history[:, :, 0].T.reshape(-1),
            "bias_theta2": error_history[:, :, 1].T.reshape(-1),
            "logdet_gain": selection_gain_history.T.reshape(-1),
        }
    )
    summary_by_step = summarize_steps(theta_true, theta_history)
    return records, summary_by_step

In [6]:
cfg = Config(
    test_length=40,
    bank_id=1,
    rho=0.0,
    n_items=150,
    testing_size=0,
    theta_csv="",
    seed=20260430,
    map_tolerance=1e-8,
    map_max_iterations=50,
    output_suffix="_python",
)

if cfg.bank_id < 1:
    raise ValueError("bank_id must be at least 1.")
if cfg.test_length < 1:
    raise ValueError("test_length must be at least 1.")
if cfg.n_items < 1:
    raise ValueError("n_items must be at least 1.")
if cfg.testing_size < 0:
    raise ValueError("testing_size cannot be negative.")
if cfg.rho not in RHO_TAGS:
    raise ValueError(f"rho must be one of {tuple(RHO_TAGS)}.")

rho_tag = RHO_TAGS[cfg.rho]
bank_path = (
    EXP_DIR / "data" / "item_banks" / f"item_bank_uncor_{cfg.bank_id}.csv"
)
item_bank_frame = pd.read_csv(bank_path)
required_item_columns = ["a1", "a2", "b"]
missing_item_columns = set(required_item_columns) - set(item_bank_frame.columns)
if missing_item_columns:
    raise ValueError(
        f"Item bank is missing required columns: {sorted(missing_item_columns)}"
    )
item_bank = item_bank_frame[required_item_columns].to_numpy(
    dtype=np.float64
)[: cfg.n_items]

if cfg.theta_csv:
    theta_path = Path(cfg.theta_csv).expanduser()
    if not theta_path.is_absolute():
        theta_path = ROOT / theta_path
else:
    theta_path = (
        EXP_DIR
        / "data"
        / "theta_true"
        / f"theta_true_{rho_tag}_{cfg.bank_id}.csv"
    )
theta_frame = pd.read_csv(theta_path)
required_theta_columns = ["theta1", "theta2"]
missing_theta_columns = set(required_theta_columns) - set(theta_frame.columns)
if missing_theta_columns:
    raise ValueError(
        f"Theta file is missing required columns: {sorted(missing_theta_columns)}"
    )
theta_true = theta_frame[required_theta_columns].to_numpy(dtype=np.float64)
if cfg.testing_size > 0:
    theta_true = theta_true[: cfg.testing_size]

if cfg.n_items > len(item_bank_frame):
    raise ValueError("n_items cannot exceed the number of items in the bank.")
if cfg.test_length > len(item_bank):
    raise ValueError("test_length cannot exceed the number of selected items.")
if len(theta_true) < 2:
    raise ValueError("At least two theta values are required for correlation.")
if not np.isfinite(item_bank).all() or not np.isfinite(theta_true).all():
    raise ValueError("Input data must contain only finite values.")

print(f"item bank  : {item_bank.shape} ({bank_path})")
print(f"theta_true : {theta_true.shape} ({theta_path})")
print(f"Config     : {cfg}")

item bank  : (150, 3) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/data/item_banks/item_bank_uncor_1.csv)
theta_true : (5000, 2) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/data/theta_true/theta_true_rho00_1.csv)
Config     : Config(test_length=40, bank_id=1, rho=0.0, n_items=150, testing_size=0, theta_csv='', seed=20260430, map_tolerance=1e-08, map_max_iterations=50, output_suffix='_python')


In [7]:
records, summary_by_step = run_d_optimality(cfg, item_bank, theta_true)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
stem = (
    f"2d_2pl_uncor_{cfg.bank_id}_{rho_tag}_{cfg.n_items}items_"
    f"D_optimality_MAP{cfg.output_suffix}"
)
records_path = RESULTS_DIR / f"records_{stem}.csv"
summary_path = RESULTS_DIR / f"summary_{stem}.csv"
records.to_csv(records_path, index=False)
summary_by_step.to_csv(summary_path, index=False)

display(summary_by_step.tail(1))
print(f"Saved records to: {records_path}")
print(f"Saved summary to: {summary_path}")

step  1, rmse_theta1 0.858, rmse_theta2 0.922, rmse_overall 0.890, map_iterations 5
step  2, rmse_theta1 0.807, rmse_theta2 0.863, rmse_overall 0.835, map_iterations 5
step  3, rmse_theta1 0.784, rmse_theta2 0.795, rmse_overall 0.790, map_iterations 5
step  4, rmse_theta1 0.736, rmse_theta2 0.759, rmse_overall 0.748, map_iterations 5
step  5, rmse_theta1 0.713, rmse_theta2 0.708, rmse_overall 0.710, map_iterations 5
step  6, rmse_theta1 0.670, rmse_theta2 0.684, rmse_overall 0.677, map_iterations 5
step  7, rmse_theta1 0.647, rmse_theta2 0.654, rmse_overall 0.650, map_iterations 5
step  8, rmse_theta1 0.627, rmse_theta2 0.635, rmse_overall 0.631, map_iterations 5
step  9, rmse_theta1 0.609, rmse_theta2 0.611, rmse_overall 0.610, map_iterations 5
step 10, rmse_theta1 0.586, rmse_theta2 0.592, rmse_overall 0.589, map_iterations 5
step 11, rmse_theta1 0.576, rmse_theta2 0.579, rmse_overall 0.577, map_iterations 5
step 12, rmse_theta1 0.563, rmse_theta2 0.564, rmse_overall 0.563, map_itera

,step,Bias_theta1,RMSE_theta1,MAE_theta1,r_theta1,Bias_theta2,RMSE_theta2,MAE_theta2,r_theta2,RMSE_overall
39,40,0.00372,0.410452,0.323856,0.910301,-0.005343,0.401896,0.319166,0.916659,0.406197


Saved records to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/results/records_2d_2pl_uncor_1_rho00_150items_D_optimality_MAP_python.csv
Saved summary to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP_v5/results/summary_2d_2pl_uncor_1_rho00_150items_D_optimality_MAP_python.csv
